# 03 — Diagnostic claim (fallback)

**Lightweight, near-certain-to-finish contribution.** Instead of *fixing* over-squashing with a new layer, we
*predict where it happens* from the algebra and check that the prediction matches a trained GNN's failures.

This reuses `aiq.gnn.over_squashing_diagnostic` (already implemented in the package): it flags
`(source, target, depth)` triples where the walk-entropy `H_g(i,j) > log2(hidden_dim)` — i.e. where more path
diversity must be packed into the embedding than its width allows.

**Pivot rule:** if the trainable `QuotientMessagePassing` does not beat baselines by the day-7 checkpoint,
this becomes the paper's empirical core ("an algebraic diagnostic of over-squashing").

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from oversquash.diagnostic import bottleneck_scores, correlate_with_errors
from oversquash.data import make_neighborsmatch_tree
import torch

In [ ]:
# Build one NeighborsMatch tree and ask the algebra where the bottlenecks are.
g = torch.Generator().manual_seed(0)
tree = make_neighborsmatch_tree(radius=4, generator=g)
ei = tree.edge_index.numpy(); N = tree.x.size(0)
HIDDEN = 32
diag = bottleneck_scores(ei, num_nodes=N, hidden_dim=HIDDEN, max_depth=4)
print('#bottleneck triples:', len(diag['bottlenecks']))
print('worst (i,j,g,H_g) :', diag['max_entropy'])

## Correlate predicted bottlenecks with empirical GNN errors

Train a vanilla GCN on a batch of trees, record which (source→root) pairs it gets wrong, and compute the
precision/recall/F1 of the algebraic predictor against those errors. A high F1 is the headline number.

In [ ]:
# Predicted bottleneck pairs (collapse depth, keep (source,target)).
predicted = {(i, j) for (i, j, gg, h) in diag['bottlenecks']}

# TODO day 7 (only if pivoting): train GCN, collect misclassified root pairs as
# `empirical_errors`, then:
#   stats = correlate_with_errors(predicted, empirical_errors)
#   print(stats)  # -> precision / recall / f1
print('predicted bottleneck pairs:', len(predicted))
print('Wire in GCN errors on day 7 if the strong claim does not land.')

In [ ]:
# Figure: entropy H_g vs log2(width) across depths — shows where compression
# exceeds capacity. Useful even if the strong claim succeeds (it motivates kQ/I).
depths = sorted({gg for (_, _, gg, _) in diag['bottlenecks']})
by_depth = {d: [h for (_, _, gg, h) in diag['bottlenecks'] if gg == d] for d in depths}
fig, ax = plt.subplots(figsize=(6, 4))
for d in depths:
    ax.scatter([d] * len(by_depth[d]), by_depth[d], alpha=0.5)
ax.axhline(np.log2(HIDDEN), color='red', ls='--', label=f'log2(width)={np.log2(HIDDEN):.1f}')
ax.set_xlabel('depth $g$'); ax.set_ylabel('walk entropy $H_g(i,j)$ (bits)')
ax.set_title('Where path diversity exceeds embedding capacity'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()